# Task 2.3: Inference of Hidden States

In this notebook, we implement hidden state inference for both the ramp and step HMM models using the forward-backward algorithm. We will:

1. Simulate spike trains from both models
2. Use forward-backward algorithm to infer hidden states
3. Compare inference accuracy between smoothing and filtering
4. Analyze parameter regimes where inference works best

## Theory

We use the Forward-Backward Algorithm (FBA) via `hmm_expected_states` to compute:
- Posterior probabilities: $P(s_t | n_{1:T})$
- Log-likelihood: $\log P(n_{1:T})$

For the ramp model, we infer $\mathbb{E}[x_t | n_{1:T}]$ where $x_t = s_t/(K-1)$
For the step model, we infer $P(s_t = \text{high state} | n_{1:T})$

We'll compare smoothing (using all observations) vs filtering (using only past observations) to see how inference accuracy differs.

In [ ]:
import sys
sys.path.append('..')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from HMM_models import RampModelHMM, StepModelHMM
import inference
from HMM_inference import perform_ramp_inference, perform_step_inference

# Set random seed for reproducibility
np.random.seed(42)

# Plotting settings
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

In [ ]:
def plot_ramp_inference_example(true_ramp, inferred_ramp, title='Ramp Model Inference'):
    """Plot ground truth vs inferred ramp trajectory"""
    plt.figure(figsize=(12, 4))
    plt.plot(true_ramp, 'b-', label='True $x_t$', linewidth=2)
    plt.plot(inferred_ramp, 'r--', label='Inferred $\mathbb{E}[x_t | n_{1:T}]$', linewidth=2)
    plt.fill_between(range(len(true_ramp)), 
                    true_ramp - 0.1, true_ramp + 0.1, 
                    alpha=0.2, color='b')
    plt.xlabel('Time step')
    plt.ylabel('$x_t$')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_step_inference_example(true_tau, est_tau, P_high, title='Step Model Inference'):
    """Plot posterior probability of high state and true jump time"""
    plt.figure(figsize=(12, 4))
    plt.plot(P_high, 'r-', label='$P(s_t | n_{1:T})$', linewidth=2)
    plt.axvline(x=true_tau, color='b', linestyle='--', 
                label=f'True jump time = {true_tau}')
    plt.axvline(x=est_tau, color='r', linestyle='--', 
                label=f'Estimated jump time = {est_tau}')
    plt.axhline(y=0.5, color='k', linestyle=':', alpha=0.5)
    plt.xlabel('Time step')
    plt.ylabel('Probability')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_error_heatmap(errors, param1_vals, param2_vals, 
                      param1_name, param2_name, title):
    """Plot heatmap of inference errors across parameter space"""
    plt.figure(figsize=(10, 8))
    plt.imshow(errors, cmap='RdYlGn_r', origin='lower', aspect='auto',
               extent=[param2_vals[0], param2_vals[-1], 
                      param1_vals[0], param1_vals[-1]])
    plt.colorbar(label='Mean Absolute Error')
    plt.xlabel(param2_name)
    plt.ylabel(param1_name)
    plt.title(title)
    plt.show()

## Ramp Model Inference

Let's first look at inference for the ramp model. We'll:
1. Simulate spike trains
2. Run inference using both smoothing and filtering
3. Compare inference accuracy

In [ ]:
# Parameters for ramp model
K = 50  # number of discrete states
beta = 0.1  # drift parameter
sigma = 0.2  # diffusion parameter
dt = 0.01  # time step
T = 500  # trial duration
R_h = 30.0  # maximum firing rate
N = 10  # number of trials

# Run inference with smoothing
true_ramps, inferred_ramps_smooth, mae_smooth = perform_ramp_inference(
    K=K, beta=beta, sigma=sigma, dt=dt, T=T, R_h=R_h, N=N, use_filter=False
)

# Run inference with filtering
_, inferred_ramps_filter, mae_filter = perform_ramp_inference(
    K=K, beta=beta, sigma=sigma, dt=dt, T=T, R_h=R_h, N=N, use_filter=True
)

# Plot example trial
plot_ramp_inference_example(
    true_ramps[0], inferred_ramps_smooth[0],
    title=f'Ramp Model Inference (Smoothing)\nMAE = {mae_smooth[0]:.3f}'
)

plot_ramp_inference_example(
    true_ramps[0], inferred_ramps_filter[0],
    title=f'Ramp Model Inference (Filtering)\nMAE = {mae_filter[0]:.3f}'
)

# Compare average errors
print(f"Average MAE with smoothing: {np.mean(mae_smooth):.3f}")
print(f"Average MAE with filtering: {np.mean(mae_filter):.3f}")

### Parameter Space Exploration

Let's explore how inference accuracy varies with different parameter values. We'll focus on:
1. Drift parameter (beta)
2. Diffusion parameter (sigma)
3. Maximum firing rate (R_h)

In [ ]:
# Explore beta vs sigma
# Change these two linspaces to increase resolution and range of heatmap
# I suggest the two ranges below for best representation, but takes about 3 mins to run
# beta_vals = np.linspace(0.05, 1, 30)
# sigma_vals = np.linspace(0.005, 1, 30)
# ie use a resolution of 30
# Having a resolution of 6 shows more granular plot but should run in about 10 seconds
ramp_resolution = 6
beta_vals = np.linspace(0.05, 1, ramp_resolution)
sigma_vals = np.linspace(0.005, 1, ramp_resolution)

errors = np.zeros((len(beta_vals), len(sigma_vals)))

for i, beta in enumerate(beta_vals):
    for j, sigma in enumerate(sigma_vals):
        _, _, mae = perform_ramp_inference(
            K=K, beta=beta, sigma=sigma, dt=dt, T=T, R_h=R_h, N=N
        )
        errors[i, j] = np.mean(mae)

plot_error_heatmap(errors, beta_vals, sigma_vals,
                   'beta', 'sigma',
                   'Ramp Model Inference Error vs Parameters')

## Step Model Inference

Now let's look at inference for the step model. We'll:
1. Simulate spike trains
2. Run inference using both smoothing and filtering
3. Compare inference accuracy

In [ ]:
# Parameters for step model
m = 50  # mean jump time
r = 10  # shape parameter
dt = 0.01  # time step
T = 500  # trial duration
R_low = 5.0  # low state firing rate
R_high = 50.0  # high state firing rate
N = 10  # number of trials
exact = True  # use exact (r+1)-state model

# Run inference with smoothing
true_taus, est_taus_smooth, mae_smooth = perform_step_inference(
    m=m, r=r, dt=dt, T=T, R_low=R_low, R_high=R_high,
    N=N, exact=exact, use_filter=False
)

# Run inference with filtering
_, est_taus_filter, mae_filter = perform_step_inference(
    m=m, r=r, dt=dt, T=T, R_low=R_low, R_high=R_high,
    N=N, exact=exact, use_filter=True
)

# Plot example trial
step_hmm = StepModelHMM(m=m, r=r, dt=dt, exact=exact)
states, tau_true, spikes = step_hmm.simulate_spikes(
    n_steps=T, R_low=R_low, R_high=R_high, dt=dt
)

# Compute P_high(t) for visualization
rates = np.zeros(step_hmm.K)
rates[-1] = R_high  # high state is last state in exact model
rates[:-1] = R_low
lambdas = rates * dt
ll = inference.poisson_logpdf(spikes, lambdas)
post_probs, _ = inference.hmm_expected_states(
    pi0=np.array([1.0] + [0.0]*(step_hmm.K-1)),
    Ps=step_hmm.T,
    ll=ll,
    filter=False
)
P_high = post_probs[:, -1]  # probability of high state

plot_step_inference_example(
    tau_true, est_taus_smooth[0], P_high,
    title=f'Step Model Inference (Smoothing)\nMAE = {mae_smooth[0]:.3f}'
)

plot_step_inference_example(
    tau_true, est_taus_filter[0], P_high,
    title=f'Step Model Inference (Filtering)\nMAE = {mae_smooth[0]:.3f}'
)

# Compare average errors
print(f"Average MAE with smoothing: {np.mean(mae_smooth):.3f}")
print(f"Average MAE with filtering: {np.mean(mae_filter):.3f}")

### Parameter Space Exploration

Let's explore how inference accuracy varies with different parameter values. We'll focus on:
1. Mean jump time (m)
2. Shape parameter (r)
3. Firing rates (R_low, R_high)

In [ ]:
# Explore m vs r
# Did ranges manually here as all values in range have to be an integer
# I tried a step size of 20 which worked well (takes 1 min to load)
# MAE seems to be 0 for a lot of the parameter space...?
step_size=50
m_vals = np.arange(5,1005,2*step_size)
r_vals = np.arange(5,505,step_size)
#m_vals = np.array([25, 50, 75, 100])
#r_vals = np.array([5, 10, 15, 20])
errors = np.zeros((len(m_vals), len(r_vals)))

for i, m in enumerate(m_vals):
    for j, r in enumerate(r_vals):
        _, _, mae = perform_step_inference(
            m=m, r=r, dt=dt, T=T, R_low=R_low, R_high=R_high,
            N=5, exact=True
        )
        errors[i, j] = np.mean(mae)

plot_error_heatmap(errors, m_vals, r_vals,
                   'm', 'r',
                   'Step Model Inference Error vs Parameters')

## Summary and Discussion

### Key Findings

1. **Smoothing vs Filtering**
   - Smoothing generally provides better inference accuracy since it uses all observations
   - Filtering is more appropriate for online/real-time inference
   - The difference in accuracy is more pronounced for the ramp model

2. **Parameter Dependence**
   - Ramp Model:
     - Higher beta (drift) generally improves inference
     - Low sigma (diffusion) works best for low errors
     - Higher firing rates help with inference accuracy?
   
   - Step Model:
     - Larger m (mean jump time) makes inference harder but more accurate?
     - Higher r (shape parameter) improves inference
     - Larger difference between R_low and R_high helps?

3. **Model-Specific Challenges**
   - Ramp Model: Inference is harder when the trajectory is noisy (high sigma) or slow (low beta)
   - Step Model: Inference is harder when jumps are rare (high m) or when m/r is about 3 (should look at NB model as to why)

### Next

1. Explore more parameter combinations
2. Analyze the effect of trial duration (T) on inference accuracy
3. Study the impact of number of trials (N) on parameter estimation
4. Compare with other inference methods 

# Improvements

Keeping these separate for now just in case

In [ ]:
#%run ../testing_task2/task2_3_improvements.py

In [ ]:
#from task2_3_improvements import evaluate_inference, analyze_parameter_regime, compare_smoothing_filtering, plot_inference_with_confidence, analyze_observation_duration, plot_duration_analysis, plot_parameter_analysis, compute_simulation_confidence_intervals

import testing_task2.task2_3_improvements as t23
import task2_3_old as t_old



In [ ]:
# 1: Compare smoothing vs filtering
t23.compare_smoothing_filtering(model_type='ramp', n_trials=10) 
t23.compare_smoothing_filtering(model_type='step', n_trials=10) 

In [ ]:
# 2: Parameter regime analysis

# These do similar to the heatmaps if needed. Hopefully it works for any parameter we want to look at


'''
Parameters used previously for reference:
ramp_resolution = 6
beta_vals = np.linspace(0.05, 1, ramp_resolution)
sigma_vals = np.linspace(0.005, 1, ramp_resolution)
step_size = 50
m_vals = np.arange(5,1005,2*step_size)
r_vals = np.arange(5,505,step_size)
'''
ramp_resolution_2 = 20
beta_vals_2 = np.linspace(0.05, 1, ramp_resolution_2)
sigma_vals_2 = np.linspace(0.005, 1, ramp_resolution_2)

step_size_2 = 20
m_vals_2 = np.arange(5,1005,2*step_size_2)
r_vals_2 = np.arange(5,505,step_size_2)


In [ ]:
# Inference accuracy vs Beta

param_range, errors = t23.analyze_parameter_regime(
    model_type='ramp',
    param_name='beta',
    param_range=beta_vals_2
)
t23.plot_parameter_analysis(param_range, errors, 'Beta', 'Ramp')


In [ ]:
# Inference accuracy vs sigma

param_range, errors = t23.analyze_parameter_regime(
    model_type='ramp',
    param_name='sigma',
    param_range=sigma_vals_2
)
t23.plot_parameter_analysis(param_range, errors, 'Sigma', 'Ramp')

In [ ]:
# Inference accuracy vs m

param_range, errors = t23.analyze_parameter_regime(
    model_type='step',
    param_name='m',
    param_range=m_vals_2
)
t23.plot_parameter_analysis(param_range, errors, 'm', 'Step')

In [ ]:
# Inference accuracy vs r

param_range, errors = t23.analyze_parameter_regime(
    model_type='step',
    param_name='r',
    param_range=r_vals_2
)
t23.plot_parameter_analysis(param_range, errors, 'r', 'Step')

In [ ]:
# 3: Observation duration analysis
# Not sure if this works the way I intended, but hopefully shows the effect of accuracy for filtering scenario as the time increases
# I don't think this works though, but have kept it in just in case

durations, errors = t23.analyze_observation_duration(
        model_type='ramp',
        durations=[100, 200, 500, 1000, 2000]
    )
t23.plot_duration_analysis(durations, errors, 'Ramp')
t23.plot_duration_analysis(durations, errors, 'Step')
    

In [ ]:
# 4: Plot with confidence intervals
# I have tried to do this with 2 methods
# Method 1, I tried to get the confidence interval range per trial and plot it, however I think I have just done it for ALL trials instead, hence why it looks weird
# Method 2, I have simulated the inference many times and plotted the 95% interval, however this looks wrong as well
# It would probably be best to do this mathematically with the formulas for the distributions
# Again, I dont think this is needed, but left this in anyway.


# Define ramp‐HMM parameters
K       = 50          # number of discrete levels
beta    = 0.1         # drift parameter
sigma   = 0.2         # diffusion parameter
dt      = 0.01        # bin width (seconds)
T       = 500         # bins per trial
R_h     = 30.0        # max Poisson rate (Hz)
N       = 20          # number of trials
use_filter = False    # False = smoothing; True = filtering


model = RampModelHMM()
true_states, spikes = model.simulate()
true_ramps, inferred_ramps, MAEs = perform_ramp_inference(
    K       = K,
    beta    = beta,
    sigma   = sigma,
    dt      = dt,
    T       = T,
    R_h     = R_h,
    N       = N,
    pi0     = None,       # default: start in state 0
    use_filter = use_filter
)



lower, upper = t_old.compute_confidence_intervals(inferred_ramps[0])

t_old.plot_inference_with_confidence(
    true_states/50, 
    inferred_ramps[0],
    (lower, upper),
    title = 'Ramp Model Inference with 95% Confidence Intervals')

In [ ]:
print(inferred_ramps.shape)
print(true_states.shape)

In [ ]:
print(t23.compute_simulation_confidence_intervals(
    perform_ramp_inference,
    n_simulations=100,
    confidence_level=0.95,
    K=K, beta=beta, sigma=sigma, dt=dt, T=T,
    R_h=R_h, N=N, pi0=None, use_filter=False
))

In [ ]:
mean_signal, mean_over_trials, lower_ci, upper_ci = t23.compute_simulation_confidence_intervals(
    perform_ramp_inference,
    n_simulations=100,
    confidence_level=0.95,
    K=K, beta=beta, sigma=sigma, dt=dt, T=T,
    R_h=R_h, N=N, pi0=None, use_filter=False
)


In [ ]:
print(mean_signal.shape)
print(mean_signal[0].shape)
print(lower_ci.shape)
print(upper_ci.shape)
print(true_states.shape)
print(mean_over_trials.shape)
print(mean_over_trials[0].shape)

In [ ]:
t23.plot_inference_with_confidence(
        true_states/50, 
        mean_signal[0],
        (lower_ci, upper_ci),
        'Ramp Model Inference with 95% Confidence Intervals'
    ) 

t23.plot_inference_with_confidence(
        true_states/50, 
        mean_over_trials[0],
        (lower_ci, upper_ci),
        'Ramp Model Inference with 95% Confidence Intervals'
    ) 

### More analysis

Trying to do for x0 (ie pi0) and Rh

In [ ]:
# Inference accuracy vs Rh
Rh_resolution = 10
Rh_list = np.linspace(1, 101, Rh_resolution)

param_range, errors = t23.analyze_parameter_regime(
    model_type='ramp',
    param_name='Rh',
    param_range=Rh_list
)
t23.plot_parameter_analysis(param_range, errors, 'Rh', 'Ramp')

In [ ]:
# Build pi0_list
pi0_list = []

#needs to be the same as inside function - currently set at 50
K = 50

for k in range(K):
    pi0 = np.zeros(K)
    pi0[k] = 1.0  # Start in state k
    pi0_list.append(pi0.tolist())  # Optionally convert to list, depending on usage

pi0_list = [np.eye(K)[k] for k in range(K)] # Converts to list of arrays


In [ ]:
# Inference accuracy vs pi0

#Takes a while (2 mins)

param_range, errors = t23.analyze_parameter_regime(
    model_type='ramp',
    param_name='pi0',
    param_range=pi0_list
)




In [ ]:
t23.pi0_plot_parameter_analysis(param_range, errors, 'pi0', 'Ramp')

In [ ]:
# Inference accuracy vs R_low

R_low_resolution = 10
R_low_list = np.linspace(1, 41, R_low_resolution)

param_range, errors = t23.analyze_parameter_regime(
    model_type='step',
    param_name='R_low',
    param_range=R_low_list
)
t23.plot_parameter_analysis(param_range, errors, 'R_low', 'Step')

In [ ]:
# Inference accuracy vs R_low

R_high_resolution = 10
R_high_list = np.linspace(11, 51, R_high_resolution)

param_range, errors = t23.analyze_parameter_regime(
    model_type='step',
    param_name='R_high',
    param_range=R_high_list
)
t23.plot_parameter_analysis(param_range, errors, 'R_high', 'Step')